In [ ]:
# -*- coding: utf-8 -*-
"""
Backtest and Visualization of Wallex Arbitrage Strategy
========================================================
This module provides:
1. Plotting ratio and diff between Wallex and Nobitex prices using Plotly.
2. Backtesting a simple threshold strategy on Wallex USDT/TMN market:
   - Portfolio only includes USDT and TMN.
   - Start with all capital in USDT.
   - Trade rules:
       z = wlx - nbtx
       if z > upper → SELL USDT (convert to TMN)
       if z < lower → BUY USDT (convert to USDT)
   - Each trade size = min(balance, order_value).
   - Fee is applied on both sides of trades.
"""

import pandas as pd
import plotly.graph_objects as go


def plot_ratio_diff_plotly(df, nbtx_col="nbtx", wlx_col="wlx"):
    """
    Plot Ratio and Diff between Wallex and Nobitex prices using Plotly.

    Parameters
    ----------
    df : pd.DataFrame
        Input DataFrame with columns [nbtx_col, wlx_col].
    nbtx_col : str
        Column name for Nobitex prices.
    wlx_col : str
        Column name for Wallex prices.

    Notes
    -----
    - Ratio = Wallex / Nobitex
    - Diff = Wallex - Nobitex
    """
    # Calculate diff and ratio
    df["diff"] = df[wlx_col] - df[nbtx_col]
    df["ratio"] = df[wlx_col] / df[nbtx_col]

    fig = go.Figure()

    # Ratio line (left y-axis)
    fig.add_trace(go.Scatter(
        x=df.index,
        y=df["ratio"],
        name="Ratio",
        mode="lines",
        line=dict(color="blue")
    ))

    # Diff line (right y-axis)
    fig.add_trace(go.Scatter(
        x=df.index,
        y=df["diff"],
        name="Diff",
        mode="lines",
        line=dict(color="red"),
        yaxis="y2"
    ))

    # Layout configuration
    fig.update_layout(
        title="Ratio (left) vs Diff (right)",
        xaxis=dict(title="Time"),
        yaxis=dict(title="Ratio", side="left"),
        yaxis2=dict(title="Diff", side="right", overlaying="y"),
        legend=dict(x=0.01, y=0.99, bordercolor="Black", borderwidth=1),
        template="plotly_white",
        height=600,
        width=1100
    )

    fig.show()


def backtest_wallex_strategy(
    df,
    upper,
    lower,
    fee=0.001,
    initial_usdt=1000,
    order_value=100
):
    """
    Backtest arbitrage strategy based on Wallex - Nobitex price difference.

    Parameters
    ----------
    df : pd.DataFrame
        Must contain columns ['nbtx', 'wlx', 'diff'].
    upper : float
        Upper threshold of diff → SELL USDT.
    lower : float
        Lower threshold of diff → BUY USDT.
    fee : float
        Transaction fee (default=0.001 → 0.1%).
    initial_usdt : float
        Starting capital in USDT.
    order_value : float
        Trade size in USDT.

    Returns
    -------
    dict
        Backtest results including:
        - final portfolio value (TMN)
        - remaining balances
        - last price
        - hold value (benchmark)
        - number of trades
        - relative performance (vs hold)
    """
    usdt = initial_usdt
    tmn = 0.0
    trades = 0

    for _, row in df.iterrows():
        z = row["diff"]   # spread between Wallex and Nobitex
        y = row["wlx"]    # Wallex price

        # --- SELL USDT → convert to TMN ---
        if z > upper and usdt > 0:
            trade_usdt = min(usdt, order_value)
            tmn += trade_usdt * y * (1 - fee)
            usdt -= trade_usdt
            trades += 1

        # --- BUY USDT → convert TMN to USDT ---
        elif z < lower and tmn > 0:
            max_affordable_usdt = tmn / (y * (1 + fee))
            trade_usdt = min(order_value, max_affordable_usdt)
            if trade_usdt > 0:
                cost_tmn = trade_usdt * y * (1 + fee)
                tmn -= cost_tmn
                usdt += trade_usdt
                trades += 1

    # Final portfolio value in TMN
    last_y = df["wlx"].iloc[-1]
    final_value = tmn + usdt * last_y

    # Benchmark: simple hold (all USDT, no trading)
    hold_value = initial_usdt * last_y

    return {
        "final_value_tmn": final_value,
        "usdt_final": usdt,
        "tmn_final": tmn,
        "wlx_last_price": last_y,
        "hold_value_tmn": hold_value,
        "num_trades": trades,
        "relative_perf": final_value / hold_value,
    }


def backtest_wallex_strategy_with_log(
    df,
    upper,
    lower,
    fee=0.001,
    initial_usdt=1000,
    order_value=100
):
    """
    Backtest arbitrage strategy with detailed trade log.

    Parameters
    ----------
    df : pd.DataFrame
        Must contain columns ['nbtx', 'wlx', 'diff'].
    upper : float
        Upper threshold of diff → SELL USDT.
    lower : float
        Lower threshold of diff → BUY USDT.
    fee : float
        Transaction fee (default=0.001 → 0.1%).
    initial_usdt : float
        Starting capital in USDT.
    order_value : float
        Trade size in USDT.

    Returns
    -------
    summary : dict
        Same as backtest_wallex_strategy.
    trades_df : pd.DataFrame
        Log of each trade including time, side, size, price, fees, balances, and portfolio value.
    """
    usdt = initial_usdt
    tmn = 0.0
    trades = []

    for ts, row in df.iterrows():
        z = row["diff"]
        y = row["wlx"]

        # --- SELL USDT → convert to TMN ---
        if z > upper and usdt > 0:
            trade_usdt = min(usdt, order_value)
            tmn_gain = trade_usdt * y * (1 - fee)
            usdt -= trade_usdt
            tmn += tmn_gain
            trades.append({
                "time": ts,
                "side": "SELL",
                "usdt_traded": trade_usdt,
                "price": y,
                "fee": fee,
                "usdt_after": usdt,
                "tmn_after": tmn,
                "portfolio_value": tmn + usdt * y
            })

        # --- BUY USDT → convert TMN to USDT ---
        elif z < lower and tmn > 0:
            max_affordable_usdt = tmn * (1 - fee) / (y * (1 + fee))
            trade_usdt = min(order_value, max_affordable_usdt)
            if trade_usdt > 0:
                cost_tmn = trade_usdt * y * (1 + fee)
                tmn -= cost_tmn
                usdt += trade_usdt
                trades.append({
                    "time": ts,
                    "side": "BUY",
                    "usdt_traded": trade_usdt,
                    "price": y,
                    "fee": fee,
                    "usdt_after": usdt,
                    "tmn_after": tmn,
                    "portfolio_value": tmn + usdt * y
                })

    # Final portfolio value in TMN
    last_y = df["wlx"].iloc[-1]
    final_value = tmn + usdt * last_y

    # Benchmark: simple hold (all USDT, no trading)
    hold_value = initial_usdt * last_y

    summary = {
        "final_value_tmn": final_value,
        "usdt_final": usdt,
        "tmn_final": tmn,
        "wlx_last_price": last_y,
        "hold_value_tmn": hold_value,
        "num_trades": len(trades),
        "relative_perf": final_value / hold_value,
    }

    trades_df = pd.DataFrame(trades)
    return summary, trades_df


# =========================
# Example Usage
# =========================
if __name__ == "__main__":
    # Example dataset (commented out)
    # data = {
    #     "ts_main": pd.date_range("2025-08-18 14:40:00", periods=10, freq="min"),
    #     "nbtx": [93438, 93362, 93362, 93401, 93380, 99651, 99675, 99680, 99680, 99683],
    #     "wlx": [93117, 93148, 93145, 93123, 93139, 99600, 99600, 99697, 99723, 99603],
    # }
    # df = pd.DataFrame(data).set_index("ts_main")
    # df["diff"] = df["wlx"] - df["nbtx"]

    # Run backtest with trade log
    
    plot_ratio_diff_plotly(df)
    summary, trades_df = backtest_wallex_strategy_with_log(
        df,
        upper=200,
        lower=-200,
        order_value=100
    )
    print("Summary:", summary)
    print("Trades:\n", trades_df)


Summary: {'final_value_tmn': 112418262.31312354, 'usdt_final': 1128.6634168963137, 'tmn_final': 7.922110000003152e-65, 'wlx_last_price': 99603.0, 'hold_value_tmn': 99603000.0, 'num_trades': 4331, 'relative_perf': 1.1286634168963137}
Trades:
        time  side   usdt_traded    price    fee   usdt_after     tmn_after  \
0       122  SELL  1.000000e+02  93620.0  0.001   900.000000  9.352638e+06   
1       123  SELL  1.000000e+02  93620.0  0.001   800.000000  1.870528e+07   
2       124  SELL  1.000000e+02  93600.0  0.001   700.000000  2.805592e+07   
3       140  SELL  1.000000e+02  93610.0  0.001   600.000000  3.740756e+07   
4       159  SELL  1.000000e+02  93589.0  0.001   500.000000  4.675710e+07   
...     ...   ...           ...      ...    ...          ...           ...   
4326  38837   BUY  7.938831e-55  99590.0  0.001  1128.663417  7.922110e-53   
4327  38839   BUY  7.949127e-58  99461.0  0.001  1128.663417  7.922110e-56   
4328  38841   BUY  7.945932e-61  99501.0  0.001  1128.66